In [1]:
!pip install pyspark
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("DeliveryETLPipeline") .getOrCreate()
print("Spark Session Created")

Spark Session Created


In [3]:
orders_data = { "order_id": [1,2,3,4,5,6], "customer_id": [101,102,101,103,104,105],
              "delivery_status": [ "Delivered", "Delivered", "Delayed", "Delayed", "Delivered", "Delayed" ],
              "issue_type": [ "Late Shipment", "No Issue", "Warehouse Delay", "Weather Delay", "No Issue", "Transport Delay" ] }
orders_pd = pd.DataFrame( orders_data )
orders_pd.to_csv( "orders.csv", index=False )
print("orders.csv created")

orders.csv created


In [4]:
orders_df = spark.read.csv( "orders.csv", header=True, inferSchema=True )
print("Orders Data")
orders_df.show()

Orders Data
+--------+-----------+---------------+---------------+
|order_id|customer_id|delivery_status|     issue_type|
+--------+-----------+---------------+---------------+
|       1|        101|      Delivered|  Late Shipment|
|       2|        102|      Delivered|       No Issue|
|       3|        101|        Delayed|Warehouse Delay|
|       4|        103|        Delayed|  Weather Delay|
|       5|        104|      Delivered|       No Issue|
|       6|        105|        Delayed|Transport Delay|
+--------+-----------+---------------+---------------+



In [5]:
latest_status_data = [ (1,"Delivered"), (2,"Delivered"), (3,"Delivered"), (4,"Delayed"), (5,"Delivered"), (6,"Delayed") ]
latest_status_df = spark.createDataFrame( latest_status_data, ["order_id","latest_delivery_status"] )
print("Latest Status Updates")
latest_status_df.show()

Latest Status Updates
+--------+----------------------+
|order_id|latest_delivery_status|
+--------+----------------------+
|       1|             Delivered|
|       2|             Delivered|
|       3|             Delivered|
|       4|               Delayed|
|       5|             Delivered|
|       6|               Delayed|
+--------+----------------------+



In [6]:
etl_df = orders_df.join( latest_status_df, on="order_id", how="left" )
print("ETL Output")
etl_df.show()

ETL Output
+--------+-----------+---------------+---------------+----------------------+
|order_id|customer_id|delivery_status|     issue_type|latest_delivery_status|
+--------+-----------+---------------+---------------+----------------------+
|       6|        105|        Delayed|Transport Delay|               Delayed|
|       5|        104|      Delivered|       No Issue|             Delivered|
|       1|        101|      Delivered|  Late Shipment|             Delivered|
|       3|        101|        Delayed|Warehouse Delay|             Delivered|
|       2|        102|      Delivered|       No Issue|             Delivered|
|       4|        103|        Delayed|  Weather Delay|               Delayed|
+--------+-----------+---------------+---------------+----------------------+



In [7]:
from pyspark.sql.functions import col
delayed_orders_df = etl_df.filter( col("latest_delivery_status") == "Delayed" )
print("Delayed Orders")
delayed_orders_df.show()

Delayed Orders
+--------+-----------+---------------+---------------+----------------------+
|order_id|customer_id|delivery_status|     issue_type|latest_delivery_status|
+--------+-----------+---------------+---------------+----------------------+
|       4|        103|        Delayed|  Weather Delay|               Delayed|
|       6|        105|        Delayed|Transport Delay|               Delayed|
+--------+-----------+---------------+---------------+----------------------+



In [9]:
delayed_orders_df.write.mode( "overwrite" ).csv( "etl_delivery_output" )
print("CSV Output Saved")

CSV Output Saved


In [10]:
delayed_orders_df.write.mode( "overwrite" ).parquet( "etl_delivery_parquet" )
print("Parquet Output Saved")

Parquet Output Saved


In [11]:
etl_df.createOrReplaceTempView( "delivery_data" )
spark.sql(""" SELECT customer_id, COUNT(*) AS total_delays FROM delivery_data WHERE latest_delivery_status = 'Delayed'
GROUP BY customer_id ORDER BY total_delays DESC LIMIT 5 """).show()

+-----------+------------+
|customer_id|total_delays|
+-----------+------------+
|        103|           1|
|        105|           1|
+-----------+------------+



In [12]:
!ls etl_delivery_output

part-00000-5a61563d-8ee9-4ffc-a1d7-439e01ed1133-c000.csv  _SUCCESS
part-00001-5a61563d-8ee9-4ffc-a1d7-439e01ed1133-c000.csv


In [13]:
output_df = spark.read.csv( "etl_delivery_output", header=True, inferSchema=True )
output_df.show()

+---+---+--------+---------------+--------+
|  4|103|Delayed2|  Weather Delay|Delayed4|
+---+---+--------+---------------+--------+
|  6|105| Delayed|Transport Delay| Delayed|
+---+---+--------+---------------+--------+

